
# pyEDITH Tutorial: exo-Venus in Imaging Mode

This tutorial will guide you through using pyEDITH in imaging mode to discern an early exo-Venus from a modern exo-Venus. We'll explore this science case in various filters.


## Before we start

Make sure you follow the instructions on the [Installation](https://pyedith.readthedocs.io/en/latest/installation.html) page.

## 1. Setup and Imports

First, let's import the necessary modules and set up our environment:


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from pyEDITH import parse_input, calculate_texp, calculate_snr, AstrophysicalScene, Observation, Observatory, set_verbosity,calculate_exposure_time_or_snr,Filter
from pyEDITH.units import *

# Set verbosity to INFO, showing info, warnings and errors. Other options are "warning" (warnings and errors), 
# "quiet" (only errors), and "debug" (all logs)
set_verbosity(level='info') 

# Set the necessary environment variables --> REPLACE WITH YOUR PATHS. 
# You can also open your .bashrc (or .zshrc) and type:
# export SCI_ENG_DIR="/path/to/Sci-Eng-Interface/hwo_sci_eng"
# export YIP_CORO_DIR="/path/to/yips"

# Loading HWO style package to make pretty plots
import hwostyle
hwostyle.use("light")
colors = hwostyle.palette



Since the exo-Venus theoretical spectra are in geometric albedo, we will need a function that converts that into contrast ratio as a function of phase angle, planet radius and separation.

In [ ]:
# Function to calculate contrast ratio as a function of
# geometric albedo, phase angle, planet radius and angular separation
# Astropy units will be needed for the planet radius and separation
def contrast_ratio(geometric_albedo, phase_angle, planet_radius, separation):
    phase_function = (np.sin(phase_angle) + (np.pi - phase_angle) * np.cos(phase_angle)) / np.pi
    contrast = geometric_albedo * phase_function * (planet_radius / separation).decompose().value ** 2
    return contrast

We will also need a function that calculates the projected angular separation based on the planet's semi-major axis, distance, eccentricity, orbital inclination, argument of periastron and true anomaly. We define this function below.

In [ ]:
def projected_angular_separation(
    a_au: float,
    d_pc: float,
    ecc: float,
    inc_deg: float,
    omega_deg: float,
    nu_deg: float,
) -> float:
    """Calculates the instantaneous projected angular separation of an exoplanet.

    Parameters:
    a_au      : Semi-major axis (Astronomical Units, AU)
    d_pc      : Distance to host star (Parsecs, pc)
    ecc       : Orbital eccentricity (0 <= ecc < 1)
    inc_deg   : Orbital inclination in degrees (0 = face-on, 90 = edge-on)
    omega_deg : Argument of periastron in degrees
    nu_deg    : True anomaly (position in orbit) in degrees

    Returns:
    Projected angular separation in arcseconds (arcsec)
    """
    # Convert angles from degrees to radians
    inc = np.radians(inc_deg)
    omega = np.radians(omega_deg)
    nu = np.radians(nu_deg)

    # 1. Calculate the instantaneous physical separation r(t) in AU
    r_au = (a_au * (1 - ecc**2)) / (1 + ecc * np.cos(nu))

    # 2. Compute 3D position vector in orbital plane coordinates
    # Primary axis aligned with line of nodes
    u = omega + nu

    # 3. Apply projection geometry (face-on vs edge-on tilt)
    # Projected physical separation in the sky plane (in AU)
    r_proj_au = r_au * np.sqrt((np.cos(u)) ** 2 + (np.sin(u) * np.cos(inc)) ** 2)

    # 4. Convert projected physical distance (AU) at distance (pc) to arcseconds
    theta_arcsec = r_proj_au / d_pc

    return theta_arcsec

Now we read the Venus spectra from the HWO Tools GitHub repository and plot their corresponding contrast ratios.

In [ ]:
# Read spectra
BASE_URL = "https://raw.githubusercontent.com/spacetelescope/hwo-tools/main/coron_model/planets/"

early_venus = np.loadtxt(BASE_URL + 'EarlyVenus_geo_albedo.txt')
early_venus_wl = early_venus[:, 0]
early_venus_alb = early_venus[:, 1]
modern_venus = np.loadtxt(BASE_URL + 'Venus_geo_albedo.txt')
modern_venus_wl = modern_venus[:, 0]
modern_venus_alb = modern_venus[:, 1]

# Set Venus's parameters
ap = 0.7233 * u.au
rp = 0.95 * u.earthRad
dist = 10 * u.pc

# Calculate contrast ratios
early_venus_fpfs = contrast_ratio(early_venus_alb, np.pi / 2, rp, ap)
modern_venus_fpfs = contrast_ratio(modern_venus_alb, np.pi / 2, rp, ap)

# Define wavelength band of the filter, this will be used for calculations later
# but, for now, we only use it for plotting purposes
filter_name = 'UVIS/F750W'
wavelength_center = 0.75
bandwidth = 0.2
waveband = np.array([wavelength_center - bandwidth / 2, wavelength_center + bandwidth / 2])

plt.plot(early_venus_wl, early_venus_fpfs, label='Early Venus')
plt.plot(modern_venus_wl, modern_venus_fpfs, label='Modern Venus')
plt.axvspan(xmin=waveband[0], xmax=waveband[1], color='k', alpha=0.1)
plt.xlabel(r'Wavelength ($\mu$m)')
plt.ylabel('Fp/Fs')
plt.legend()
plt.tight_layout()

### 2. Defining Input Parameters

Let's set up the parameters for a Venus-like planet around a Sun-like star. Our objective is to estimate the exposure time to differentiate between modern and early exo-Venus. To do that, first we need to estimate the required SNR.

In [ ]:
# Handy numpy array slicing to work with the model within the filter bandpass
sect_early = np.logical_and(early_venus_wl > waveband[0], early_venus_wl < waveband[1])
sect_modern = np.logical_and(modern_venus_wl > waveband[0], modern_venus_wl < waveband[1])

# Average contrast ratios inside the wavelength band
Fp_Fs_band_earlyVenus = np.mean(early_venus_fpfs[sect_early])
Fp_Fs_band_modernVenus = np.mean(modern_venus_fpfs[sect_modern])

# Ratio between modern and early Venus
ratio_modern_early = Fp_Fs_band_modernVenus / Fp_Fs_band_earlyVenus
print('Early and modern Venus differ by a factor of {:.2f} in Fp/Fs in the band {}-{} micron.'.format(ratio_modern_early, waveband[0], waveband[1]))

# Required SNR for a 5-sigma differentiation between modern and early Venus
required_snr_5sigma = 5 / (ratio_modern_early - 1)
print('The required SNR to differ between Modern and Early Venus with 5-sigma confidence is thus roughly {:.2f}.'.format(required_snr_5sigma))

### 1.3 Running the ETC

We can now calculate the exposure time. 

In [ ]:
# Calculate separation in arcsec
separation = projected_angular_separation(ap.value, dist.value, ecc=0.0, inc_deg=45.0, omega_deg=0.0, nu_deg=0.)

imaging_params = {
    'wavelength': wavelength_center,              # Wavelength in microns
    'snr': required_snr_5sigma,                       # Desired signal-to-noise ratio
    'CRb_multiplier': 2.0,          # Count rate ratio multiplier (assuming differential imaging for PSF subtraction)
    'psf_trunc_ratio': 0.3,         # PSF Truncation Ratio to calculate photometric aperture of solid angle Omega. 
    'distance': dist.value,                 # Distance to star in parsecs
    'FstarV_10pc': 122.9279,        # Stellar flux at 10 pc in the V band [ph/cm2/s/nm]
    'Fstar_10pc': 115.59984,        # Stellar flux at 10 pc in the observed band [ph/cm2/s/nm]
    'Fp/Fs': Fp_Fs_band_earlyVenus,                # Planet-to-star contrast
    'stellar_radius': 1,            # Stellar radius in solar radii
    'nzodis': 3.0,                  # Number of zodiacal light disks
    'ra': 236.00757736823,          # Right ascension of star [deg]
    'dec': 2.51516683165,           # Declination of star [deg]
    'separation': separation,              # Separation between star and planet in arcseconds
    'observatory_preset': 'EAC1',   # Preset observatory configuration
    'observing_mode': 'IMAGER',     # Observing mode
    'filter_list': Filter(filter_name, 
                          center=wavelength_center, 
                          bandwidth=bandwidth, 
                          type="IMAGER") # filter in which to perform the observation
}


In [ ]:
# Make the parameters be the shape that the code desires 
parsed_parameters= parse_input.parse_parameters(imaging_params)

In [ ]:
# Calculate Exposure time
texp, validation_output = calculate_texp(imaging_params)
for filter_name, result in texp.items():
    print(f"Filter: {filter_name}")
    print(f"  Wavelength: {result['wavelength']}")
    print(f"  Calculated exposure time: {result['exposure_time'].to(u.hr)}")

`validation_output` contains some interesting quantities that you can use to double check or validate, or to plot additional quantities.

In [ ]:
validation_output